# 07 — Promotion & Holiday Calendar

## 1. Introduction and objectives

The CallVerse forecast formula, `orders(t) × contact-rate(intent | order state, promo, delay)`, needs to know **when promotions happen**. Olist does not record its campaigns. This notebook (Phase 1E-d) builds a calendar of promotion periods and public holidays, checks it, and describes how it relates to Olist activity.

**Design principles**

| Principle | How it is applied |
|---|---|
| **Coherent with Olist** | Dates come from the real Brazilian retail calendar (Black Friday, Christmas, Mother's Day, …) and national holidays, not random synthetic campaigns. *Assumption:* Olist campaigns coincided with these dates. |
| **Fixed in advance** | Events are defined from the public calendar in `configs/data.yaml`, **not** chosen because the data show a peak. Selecting events by their effect would leak the target into future forecasts. |
| **No double counting** | Promotions already raise Olist orders, and contacts are generated from orders. Version 1 adds **no extra promotion multiplier** to contacts. |
| **Imbalance made explicit** | Promotion and holiday days are rare, and most events occur only once or twice. The notebook measures this and recommends grouped features. |
| **Checked automatically** | Weekdays, holiday rules, overlaps and day coverage are verified before export. |

## 2. Imports, paths and inputs

In [1]:
from pathlib import Path
import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_FILE = PROJECT_ROOT / "configs" / "data.yaml"

required_files = [CONFIG_FILE, PROCESSED_DIR / "olist_daily_activity.parquet", PROCESSED_DIR / "olist_orders_analytical.parquet"]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing inputs: {missing_files}. Run notebooks 03 and 04 first.")

with open(CONFIG_FILE, encoding="utf-8") as config_file:
    calendar_config = yaml.safe_load(config_file)["generation"]["calendar"]

daily_activity = pd.read_parquet(PROCESSED_DIR / "olist_daily_activity.parquet")
orders = pd.read_parquet(PROCESSED_DIR / "olist_orders_analytical.parquet")

FIRST_DAY = daily_activity["date"].min()
LAST_DAY = daily_activity["date"].max()
print("Calendar version:", calendar_config["version"])
print("Promotion events:", len(calendar_config["promo_events"]), "| public holidays:", len(calendar_config["public_holidays"]))
print("Olist daily table covers", FIRST_DAY.date(), "to", LAST_DAY.date())

Calendar version: 1
Promotion events: 7 | public holidays: 26
Olist daily table covers 2016-09-04 to 2018-10-17


## 3. Build the event table

Each promotion event has one or more **event days** (for example Black Friday 2016 and 2017) and a **window** around each day:

- window start = event day − `days_before`
- window end = event day + `days_after`

For gift events the window is the week **before** the day (`days_after = -1` ends the window the day before), because gifts are bought in advance. Black Friday's window runs from the Friday to Cyber Monday.

In [2]:
event_rows = []
for event in calendar_config["promo_events"]:
    for event_date in event["dates"]:
        event_day = pd.Timestamp(event_date)
        event_rows.append({
            "event_id": f"{event['name']}_{event_day.year}",
            "event_name": event["name"],
            "event_group": event["group"],
            "event_day": event_day,
            "expected_weekday": event["weekday"],
            "window_start": event_day - pd.Timedelta(days=event["days_before"]),
            "window_end": event_day + pd.Timedelta(days=event["days_after"]),
        })
events = pd.DataFrame(event_rows).sort_values("window_start").reset_index(drop=True)
events["window_days"] = (events["window_end"] - events["window_start"]).dt.days + 1
events["event_weekday"] = events["event_day"].dt.day_name()

holidays = pd.DataFrame(calendar_config["public_holidays"])
holidays["date"] = pd.to_datetime(holidays["date"])

display(events[["event_id", "event_group", "event_day", "event_weekday", "window_start", "window_end", "window_days"]])

,event_id,event_group,event_day,event_weekday,window_start,window_end,window_days
0,childrens_day_2016,gift_season,2016-10-12,Wednesday,2016-10-05,2016-10-11,7
1,black_friday_2016,major_sale,2016-11-25,Friday,2016-11-25,2016-11-28,4
2,christmas_2016,gift_season,2016-12-25,Sunday,2016-12-01,2016-12-23,23
3,consumer_day_2017,major_sale,2017-03-15,Wednesday,2017-03-13,2017-03-15,3
4,mothers_day_2017,gift_season,2017-05-14,Sunday,2017-05-07,2017-05-13,7
5,valentines_day_br_2017,gift_season,2017-06-12,Monday,2017-06-05,2017-06-11,7
6,fathers_day_2017,gift_season,2017-08-13,Sunday,2017-08-06,2017-08-12,7
7,childrens_day_2017,gift_season,2017-10-12,Thursday,2017-10-05,2017-10-11,7
8,black_friday_2017,major_sale,2017-11-24,Friday,2017-11-24,2017-11-27,4
9,christmas_2017,gift_season,2017-12-25,Monday,2017-12-01,2017-12-23,23


## 4. Coherence checks

| Check | Why |
|---|---|
| Event days fall on the expected weekday | Black Friday must be a Friday; Mother's and Father's Day in Brazil are Sundays |
| Mother's / Father's Day are the 2nd Sunday of May / August | Brazilian rule |
| Each window has at least one day and starts on or before its end | A reversed window would be meaningless |
| Promotion windows do not overlap | Each day belongs to at most one promotion, so the calendar is unambiguous |
| Holiday dates are unique and within the Olist range | No duplicated or out-of-range holidays |
| Easter-based holidays match Easter | Carnival = Easter − 48/−47 days, Good Friday = Easter − 2, Corpus Christi = Easter + 60 |

Easter is computed with the standard Gregorian algorithm (Anonymous Gregorian / Meeus), so the hard-coded dates in the config are verified independently.

In [3]:
def easter_sunday(year):
    # Anonymous Gregorian algorithm (Meeus/Jones/Butcher)
    a = year % 19
    b, c = divmod(year, 100)
    d, e = divmod(b, 4)
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i, k = divmod(c, 4)
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    month = (h + l - 7 * m + 114) // 31
    day = (h + l - 7 * m + 114) % 31 + 1
    return pd.Timestamp(year=year, month=month, day=day)

checks = []

# Weekdays
wrong_weekday = events[events["expected_weekday"].notna() & (events["event_weekday"] != events["expected_weekday"])]
checks.append(("Event days fall on the expected weekday", wrong_weekday.empty))

# 2nd Sunday rule for Mother's Day (May) and Father's Day (August)
second_sunday_ok = True
for _, event in events[events["event_name"].isin(["mothers_day", "fathers_day"])].iterrows():
    day = event["event_day"]
    is_second_sunday = day.dayofweek == 6 and 8 <= day.day <= 14
    expected_month = 5 if event["event_name"] == "mothers_day" else 8
    if not (is_second_sunday and day.month == expected_month):
        second_sunday_ok = False
checks.append(("Mother's/Father's Day are the 2nd Sunday of May/August", second_sunday_ok))

# Windows
checks.append(("Every window starts on or before its end", bool((events["window_days"] >= 1).all())))
overlaps = events["window_start"].iloc[1:].values <= events["window_end"].iloc[:-1].values
checks.append(("Promotion windows do not overlap", not overlaps.any()))

# Holidays
checks.append(("Holiday dates are unique", holidays["date"].is_unique))
checks.append(("Holidays lie within the Olist date range", bool(holidays["date"].between(FIRST_DAY, LAST_DAY).all())))

# Easter-based holidays
easter_offsets = {"carnival_monday": -48, "carnival_tuesday": -47, "good_friday": -2, "corpus_christi": 60}
easter_ok = True
for _, holiday in holidays[holidays["name"].isin(easter_offsets)].iterrows():
    expected_date = easter_sunday(holiday["date"].year) + pd.Timedelta(days=easter_offsets[holiday["name"]])
    if holiday["date"] != expected_date:
        easter_ok = False
        print("Mismatch:", holiday["name"], holiday["date"].date(), "expected", expected_date.date())
checks.append(("Easter-based holidays match the computed Easter", easter_ok))

print("Easter 2017:", easter_sunday(2017).date(), "| Easter 2018:", easter_sunday(2018).date())
checks_df = pd.DataFrame(checks, columns=["check", "passed"])
display(checks_df)
if not checks_df["passed"].all():
    raise ValueError("A calendar check failed; fix configs/data.yaml before continuing.")

Easter 2017: 2017-04-16 | Easter 2018: 2018-04-01


,check,passed
0,Event days fall on the expected weekday,True
1,Mother's/Father's Day are the 2nd Sunday of Ma...,True
2,Every window starts on or before its end,True
3,Promotion windows do not overlap,True
4,Holiday dates are unique,True
5,Holidays lie within the Olist date range,True
6,Easter-based holidays match the computed Easter,True


## 5. Daily calendar table

One row per day of the Olist daily table, with:

| Column | Meaning |
|---|---|
| `is_public_holiday`, `holiday_name` | National public holiday |
| `in_promo_window`, `promo_event_id`, `promo_event_name`, `promo_group` | Day inside a promotion window (empty otherwise) |
| `is_promo_event_day` | The event day itself (for example Black Friday) |
| `days_to_next_event_day` | Days until the next promotion event day. Known in advance, so usable by forecasts. Empty after the last event. |

All columns depend only on the public calendar, never on Olist outcomes, so they can be used at any forecast horizon.

In [4]:
daily_calendar = pd.DataFrame({"date": daily_activity["date"]})

holiday_name_by_date = holidays.set_index("date")["name"]
daily_calendar["holiday_name"] = daily_calendar["date"].map(holiday_name_by_date)
daily_calendar["is_public_holiday"] = daily_calendar["holiday_name"].notna()

daily_calendar["promo_event_id"] = pd.NA
daily_calendar["promo_event_name"] = pd.NA
daily_calendar["promo_group"] = pd.NA
for _, event in events.iterrows():
    in_window = daily_calendar["date"].between(event["window_start"], event["window_end"])
    daily_calendar.loc[in_window, "promo_event_id"] = event["event_id"]
    daily_calendar.loc[in_window, "promo_event_name"] = event["event_name"]
    daily_calendar.loc[in_window, "promo_group"] = event["event_group"]
daily_calendar["in_promo_window"] = daily_calendar["promo_event_id"].notna()
daily_calendar["is_promo_event_day"] = daily_calendar["date"].isin(events["event_day"])

# Days until the next event day (searchsorted finds the first event day on or after each date)
event_days = events["event_day"].sort_values().reset_index(drop=True)
next_index = event_days.searchsorted(daily_calendar["date"])
next_event_day = pd.Series(pd.NaT, index=daily_calendar.index, dtype="datetime64[ns]")
has_next = next_index < len(event_days)
next_event_day[has_next] = event_days.iloc[next_index[has_next]].values
daily_calendar["days_to_next_event_day"] = (next_event_day - daily_calendar["date"]).dt.days.astype("Int64")

daily_calendar["in_analysis_window"] = daily_activity["in_analysis_window"].values
for column in ["promo_event_id", "promo_event_name", "promo_group", "holiday_name"]:
    daily_calendar[column] = daily_calendar[column].astype("string")

print("Daily calendar:", daily_calendar.shape)
display(daily_calendar[daily_calendar["in_promo_window"]].head(8))

Daily calendar: (774, 10)


,date,holiday_name,is_public_holiday,promo_event_id,promo_event_name,promo_group,in_promo_window,is_promo_event_day,days_to_next_event_day,in_analysis_window
31,2016-10-05,<NA>,False,childrens_day_2016,childrens_day,gift_season,True,False,7,False
32,2016-10-06,<NA>,False,childrens_day_2016,childrens_day,gift_season,True,False,6,False
33,2016-10-07,<NA>,False,childrens_day_2016,childrens_day,gift_season,True,False,5,False
34,2016-10-08,<NA>,False,childrens_day_2016,childrens_day,gift_season,True,False,4,False
35,2016-10-09,<NA>,False,childrens_day_2016,childrens_day,gift_season,True,False,3,False
36,2016-10-10,<NA>,False,childrens_day_2016,childrens_day,gift_season,True,False,2,False
37,2016-10-11,<NA>,False,childrens_day_2016,childrens_day,gift_season,True,False,1,False
82,2016-11-25,<NA>,False,black_friday_2016,black_friday,major_sale,True,True,0,False


In [5]:
# Coverage checks on the daily table
expected_window_days = 0
for event in events.itertuples():
    expected_window_days += int(daily_calendar["date"].between(event.window_start, event.window_end).sum())

calendar_checks = [
    ("Same dates as the Olist daily table", daily_calendar["date"].equals(daily_activity["date"])),
    ("Dates are unique", daily_calendar["date"].is_unique),
    ("Every holiday in the config appears exactly once", int(daily_calendar["is_public_holiday"].sum()) == len(holidays)),
    ("Promotion-window days match the event table (no day counted twice)",
     int(daily_calendar["in_promo_window"].sum()) == expected_window_days),
]
calendar_checks_df = pd.DataFrame(calendar_checks, columns=["check", "passed"])
display(calendar_checks_df)
if not calendar_checks_df["passed"].all():
    raise ValueError("A daily-calendar check failed.")

,check,passed
0,Same dates as the Olist daily table,True
1,Dates are unique,True
2,Every holiday in the config appears exactly once,True
3,Promotion-window days match the event table (n...,True


## 6. Imbalance

A forecasting model learns from examples. If a flag is `True` on very few days, or an event happens only once, the model has almost nothing to learn from and may **overfit** (memorise one spike). We therefore measure how rare each flag is inside the analysis window and how many times each event occurs.

In [6]:
window_calendar = daily_calendar[daily_calendar["in_analysis_window"]]
window_day_count = len(window_calendar)

imbalance_rows = [
    {"flag": "in_promo_window", "true_days": int(window_calendar["in_promo_window"].sum())},
    {"flag": "is_promo_event_day", "true_days": int(window_calendar["is_promo_event_day"].sum())},
    {"flag": "is_public_holiday", "true_days": int(window_calendar["is_public_holiday"].sum())},
]
for group in sorted(events["event_group"].unique()):
    imbalance_rows.append({"flag": f"promo_group = {group}", "true_days": int((window_calendar["promo_group"] == group).sum())})
imbalance_df = pd.DataFrame(imbalance_rows)
imbalance_df["percent_of_window_days"] = (imbalance_df["true_days"] / window_day_count * 100).round(2)
print("Days in the analysis window:", window_day_count)
display(imbalance_df)

window_start, window_end = window_calendar["date"].min(), window_calendar["date"].max()
events_in_window = events[(events["window_end"] >= window_start) & (events["window_start"] <= window_end)]
occurrences = events_in_window.groupby(["event_group", "event_name"]).size().rename("occurrences_in_window").reset_index()
display(occurrences)

Days in the analysis window: 608


,flag,true_days,percent_of_window_days
0,in_promo_window,82,13.49
1,is_promo_event_day,11,1.81
2,is_public_holiday,19,3.12
3,promo_group = gift_season,72,11.84
4,promo_group = major_sale,10,1.64


,event_group,event_name,occurrences_in_window
0,gift_season,childrens_day,1
1,gift_season,christmas,1
2,gift_season,fathers_day,2
3,gift_season,mothers_day,2
4,gift_season,valentines_day_br,2
5,major_sale,black_friday,1
6,major_sale,consumer_day,2


**Reading the imbalance**

- Promotion windows and holidays cover only a small share of days. That is normal for calendar effects, but it means each flag has few positive examples.
- **Each event occurs at most twice** in the analysis window, and Black Friday only once (2017). A separate flag per event would let a model memorise single spikes.

**Recommendation for the forecasting phase**

1. Use **grouped** features (`promo_group` = `major_sale` / `gift_season`, `is_public_holiday`) instead of one flag per event.
2. Treat Black Friday as a **known outlier**: give it its own simple adjustment or exclude it from model training and evaluate it separately, rather than letting it drive the fit.
3. Evaluate with **rolling backtests** that include at least one promotion window in each test period, so the error on promotion days is measured and not averaged away.

Rebalancing techniques such as oversampling do **not** apply here. This is a time series, and duplicating days would break its time order.

## 7. How events relate to Olist activity (descriptive only)

For each event inside the analysis window we compare the average daily orders **during the window** with a **baseline**: the average over the 28 days before the window, excluding other promotion and holiday days. We do the same for the share of orders delivered late, because late deliveries drive contacts (notebook 06).

This is descriptive. It does **not** decide which events are kept: all events stay in the calendar whatever their uplift (section 1).

In [7]:
orders_per_day = daily_activity.set_index("date")["orders_purchased"]
calendar_by_date = daily_calendar.set_index("date")
window_orders = orders[orders["in_analysis_window"]]
delivered_window_orders = window_orders[window_orders["has_customer_delivery_timestamp"]]

uplift_rows = []
for event in events_in_window.itertuples():
    baseline_days = pd.date_range(event.window_start - pd.Timedelta(days=28), event.window_start - pd.Timedelta(days=1))
    baseline_days = baseline_days[baseline_days >= window_start]
    normal_days = [day for day in baseline_days
                   if not calendar_by_date.loc[day, "in_promo_window"] and not calendar_by_date.loc[day, "is_public_holiday"]]
    event_days_range = pd.date_range(max(event.window_start, window_start), min(event.window_end, window_end))
    if len(normal_days) < 7 or len(event_days_range) == 0:
        continue  # not enough baseline days (e.g. right at the start of the window)

    baseline_orders = orders_per_day.loc[normal_days].mean()
    event_orders = orders_per_day.loc[event_days_range].mean()

    in_event = delivered_window_orders["purchase_date"].between(event_days_range[0], event_days_range[-1])
    in_baseline = delivered_window_orders["purchase_date"].isin(normal_days)
    late_event = delivered_window_orders.loc[in_event, "delivered_after_estimated_date"].mean()
    late_baseline = delivered_window_orders.loc[in_baseline, "delivered_after_estimated_date"].mean()

    uplift_rows.append({
        "event_id": event.event_id, "event_group": event.event_group,
        "baseline_days": len(normal_days), "event_days": len(event_days_range),
        "baseline_orders_per_day": round(baseline_orders, 1), "event_orders_per_day": round(event_orders, 1),
        "order_uplift_percent": round((event_orders / baseline_orders - 1) * 100, 1),
        "late_percent_baseline": round(late_baseline * 100, 2), "late_percent_event": round(late_event * 100, 2),
    })
uplift_df = pd.DataFrame(uplift_rows)
display(uplift_df)
print("Events skipped for lack of baseline days:", sorted(set(events_in_window["event_id"]) - set(uplift_df["event_id"])))

,event_id,event_group,baseline_days,event_days,baseline_orders_per_day,event_orders_per_day,order_uplift_percent,late_percent_baseline,late_percent_event
0,consumer_day_2017,major_sale,26,3,68.1,106.0,55.6,3.35,5.67
1,mothers_day_2017,gift_season,25,7,87.8,113.3,29.0,6.04,3.88
2,valentines_day_br_2017,gift_season,22,7,121.5,121.0,-0.4,2.56,2.21
3,fathers_day_2017,gift_season,28,7,136.4,136.0,-0.3,2.98,3.02
4,childrens_day_2017,gift_season,27,7,147.3,148.1,0.5,4.26,4.60
5,black_friday_2017,major_sale,26,4,171.4,617.2,260.1,7.88,18.29
6,christmas_2017,gift_season,23,23,202.3,206.0,1.8,9.96,8.31
7,consumer_day_2018,major_sale,27,3,249.3,238.7,-4.3,21.73,21.93
8,mothers_day_2018,gift_season,26,7,239.8,284.4,18.6,3.75,6.33
9,valentines_day_br_2018,gift_season,22,7,189.7,215.3,13.5,6.56,1.81


Events skipped for lack of baseline days: []


**Reading the table.**

- Black Friday shows by far the largest order uplift. Orders placed during it are also delivered late much more often, which, through the notebook 06 assumptions, leads to more contacts weeks later.
- Gift events show smaller and less consistent effects. Some even fall *below* their baseline, which is a reason not to assume a fixed promotion boost.
- These effects arise from Olist orders themselves, which is why version 1 adds no promotion multiplier to contacts.
- **Caution:** the baseline is the 28 days before each window and does not adjust for Olist's growth. In early 2017 orders were growing fast, so uplifts such as Consumer Day 2017 are probably overstated. The numbers are indicative only.

## 8. Export

In [8]:
events_export = events[["event_id", "event_name", "event_group", "event_day", "event_weekday",
                        "window_start", "window_end", "window_days"]]
events_export.to_csv(PROCESSED_DIR / "promo_calendar_events.csv", index=False)
daily_calendar.to_parquet(PROCESSED_DIR / "daily_calendar.parquet", index=False)

reloaded = pd.read_parquet(PROCESSED_DIR / "daily_calendar.parquet")
print("promo_calendar_events.csv:", len(events_export), "events")
print("daily_calendar.parquet:", reloaded.shape, "| same types:", reloaded.dtypes.equals(daily_calendar.dtypes))

promo_calendar_events.csv: 14 events
daily_calendar.parquet: (774, 10) | same types: True


## 9. Summary and limitations

**Produced**

- `promo_calendar_events.csv`: one row per event occurrence with its window.
- `daily_calendar.parquet`: one row per day with holiday, promotion-window, event-day and days-to-next-event columns, aligned with `olist_daily_activity.parquet`.

**Guarantees**

- Real calendar dates, verified for weekday, the 2nd-Sunday rule and Easter-based holidays.
- No overlapping promotion windows; one calendar row per day.
- Events fixed in advance, so there is no target leakage into forecasts.
- No double counting: promotions affect contacts only through Olist orders.

**Limitations**

- **Assumed campaigns.** Olist's real campaigns are unknown; the calendar assumes they followed the public retail calendar. Seller-specific or flash promotions are not represented.
- **Few occurrences.** At most two per event and one Black Friday in the window; use grouped features and treat Black Friday as an outlier.
- **National holidays only.** Regional holidays (state or city) are not included.
- **Window lengths are assumptions** (for example, one week before gift events).
- **Uplift is descriptive and trend-blind.** The section 7 baseline does not remove growth or seasonality.